
# 3. Demo notebook — inference on an unlabeled input CSV

This notebook loads the same single bundle file saved by the training notebook.

Input CSV requirements:
- must contain `premise` and `hypothesis` columns
- must not contain labels

Output CSV format:
- exactly one column named `prediction`


In [1]:
!pip -q install torch pandas numpy scikit-learn matplotlib tqdm gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 59.8 MB/s eta 0:00:00


In [2]:

import os
import re
import json
import time
import math
import random
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.optim.swa_utils import AveragedModel, SWALR, update_bn

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix,
    classification_report,
    roc_curve,
)



In [3]:
"""
ESIM-style NLI model

Core architecture:
- ESIM: Chen et al. (2017), "Enhanced LSTM for Natural Language Inference"
  https://aclanthology.org/P17-1152/

Extensions:
- learned gated pooling over the composed sequence
- sentence-level interaction features [u, v, |u-v|, u*v]
"""

from __future__ import annotations

import numpy as np
import torch
from torch import nn
from torch.nn.utils.rnn import pack_padded_sequence, pad_packed_sequence


def masked_softmax(scores: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
    """Softmax that ignores padded positions."""
    scores = scores.masked_fill(~mask, -1e9)
    return torch.softmax(scores, dim=-1)


class GatedPooling(nn.Module):
    """
    Lightweight learned pooling.

    Plain ESIM uses avg/max pooling after the composition BiLSTM.
    Here we keep avg/max information, but also learn a token importance gate.
    """
    def __init__(self, input_dim: int):
        super().__init__()
        self.gate = nn.Linear(input_dim, 1)

    def forward(self, x: torch.Tensor, mask: torch.Tensor) -> torch.Tensor:
        # x: [batch, seq_len, dim], mask: [batch, seq_len]
        mask_f = mask.unsqueeze(-1).float()

        # Average pooling over valid tokens.
        avg_pool = (x * mask_f).sum(dim=1) / mask_f.sum(dim=1).clamp(min=1e-6)

        # Max pooling over valid tokens.
        max_pool = x.masked_fill(~mask.unsqueeze(-1), -1e9).max(dim=1).values

        # Learned gated pooling.
        gate_scores = self.gate(x).squeeze(-1)                        # [batch, seq_len]
        gate_scores = gate_scores.masked_fill(~mask, -1e9)
        gate_weights = torch.softmax(gate_scores, dim=1).unsqueeze(-1)
        gated_pool = (x * gate_weights).sum(dim=1)

        return torch.cat([avg_pool, max_pool, gated_pool], dim=-1)


class ESIMPlus(nn.Module):
    """
    ESIM-style NLI encoder/classifier.

    Paper feature map
    -----------------
    Base ESIM features from Chen et al. (2017):
    - BiLSTM encoding
    - soft alignment between premise and hypothesis
    - local inference matching [x, y, x-y, x*y]
    - second BiLSTM composition layer

    Extensions:
    - learned gated pooling
    - sentence-level interaction features [u, v, |u-v|, u*v]

    Local inference vs sentence-level interaction
    ---------------------------------------------
    Local inference is token-level:
        compare each token to its aligned token summary in the other sentence.
    Sentence-level interaction is sequence-level:
        compare the final pooled premise vector u and pooled hypothesis vector v.
    """
    def __init__(
        self,
        vocab_size: int,
        embedding_dim: int,
        hidden_size: int,
        padding_idx: int = 0,
        embedding_matrix: np.ndarray | None = None,
        dropout: float = 0.3,
        train_embeddings: bool = True,
    ):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=padding_idx)
        if embedding_matrix is not None:
            self.embedding.weight.data.copy_(torch.tensor(embedding_matrix))
        self.embedding.weight.requires_grad = train_embeddings

        self.embedding_dropout = nn.Dropout(dropout)

        self.encoder = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        # ESIM local inference vectors have width 8h:
        # [x, y, x-y, x*y] and each x/y is 2h from the BiLSTM.
        self.projection = nn.Sequential(
            nn.Linear(hidden_size * 8, hidden_size),
            nn.ReLU(),
        )

        self.composition = nn.LSTM(
            input_size=hidden_size,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )

        composed_dim = hidden_size * 2
        pooled_dim = composed_dim * 3  # avg + max + gated

        self.pooler = GatedPooling(composed_dim)

        # Sentence-level interaction features:
        # u, v, |u-v|, u*v
        classifier_input_dim = pooled_dim * 4
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(classifier_input_dim, hidden_size * 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size * 2, 1),
        )

    def _run_bilstm(self, x: torch.Tensor, lengths: torch.Tensor, lstm: nn.LSTM):
        lstm.flatten_parameters()
        packed = pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_output, _ = lstm(packed)
        output, _ = pad_packed_sequence(packed_output, batch_first=True)
        return output

    def _apply_attention(
        self,
        a: torch.Tensor,
        a_mask: torch.Tensor,
        b: torch.Tensor,
        b_mask: torch.Tensor,
    ) -> tuple[torch.Tensor, torch.Tensor]:
        # Token-to-token similarity matrix.
        similarity = torch.matmul(a, b.transpose(1, 2))

        b_mask_expanded = b_mask.unsqueeze(1).expand(-1, a.size(1), -1)
        attn_a = masked_softmax(similarity, b_mask_expanded)
        attended_a = torch.matmul(attn_a, b)

        a_mask_expanded = a_mask.unsqueeze(1).expand(-1, b.size(1), -1)
        attn_b = masked_softmax(similarity.transpose(1, 2), a_mask_expanded)
        attended_b = torch.matmul(attn_b, a)

        return attended_a, attended_b

    def forward(
        self,
        premise_ids: torch.Tensor,
        premise_lengths: torch.Tensor,
        hypothesis_ids: torch.Tensor,
        hypothesis_lengths: torch.Tensor,
    ) -> torch.Tensor:
        premise_mask = premise_ids.ne(0)
        hypothesis_mask = hypothesis_ids.ne(0)

        premise_embed = self.embedding_dropout(self.embedding(premise_ids))
        hypothesis_embed = self.embedding_dropout(self.embedding(hypothesis_ids))

        premise_encoded = self._run_bilstm(premise_embed, premise_lengths, self.encoder)
        hypothesis_encoded = self._run_bilstm(hypothesis_embed, hypothesis_lengths, self.encoder)

        premise_aligned, hypothesis_aligned = self._apply_attention(
            premise_encoded, premise_mask, hypothesis_encoded, hypothesis_mask
        )

        # Local inference matching from ESIM.
        premise_enhanced = torch.cat(
            [
                premise_encoded,
                premise_aligned,
                premise_encoded - premise_aligned,
                premise_encoded * premise_aligned,
            ],
            dim=-1,
        )
        hypothesis_enhanced = torch.cat(
            [
                hypothesis_encoded,
                hypothesis_aligned,
                hypothesis_encoded - hypothesis_aligned,
                hypothesis_encoded * hypothesis_aligned,
            ],
            dim=-1,
        )

        premise_projected = self.projection(premise_enhanced)
        hypothesis_projected = self.projection(hypothesis_enhanced)

        premise_composed = self._run_bilstm(premise_projected, premise_lengths, self.composition)
        hypothesis_composed = self._run_bilstm(hypothesis_projected, hypothesis_lengths, self.composition)

        premise_vector = self.pooler(premise_composed, premise_mask)
        hypothesis_vector = self.pooler(hypothesis_composed, hypothesis_mask)

        # Sentence-level interaction features.
        pair_vector = torch.cat(
            [
                premise_vector,
                hypothesis_vector,
                torch.abs(premise_vector - hypothesis_vector),
                premise_vector * hypothesis_vector,
            ],
            dim=-1,
        )

        logits = self.classifier(pair_vector).squeeze(-1)
        return logits


In [4]:

# ---------------------------
# User configuration
# ---------------------------
INPUT_PATH = "test.csv"
MODEL_BUNDLE_PATH = "nli_esim_plus_bundle.pt"
OUTPUT_PATH = "Group_n_B.csv"


In [5]:

# Shared utility functions used for loading data and reproducing the exact tokenisation.

def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

LOWERCASE = True
TOKEN_RE = re.compile(r"\w+|[^\w\s]")

def tokenize(text: str):
    """Use the same lightweight regex tokenizer as training."""
    text = "" if pd.isna(text) else str(text)
    if LOWERCASE:
        text = text.lower()
    return TOKEN_RE.findall(text)

def encode_text(text: str, vocab: dict, max_len: int):
    """Turn one text string into token ids using the saved vocabulary."""
    token_ids = [vocab.get(tok, vocab["<unk>"]) for tok in tokenize(text)[:max_len]]
    if not token_ids:
        token_ids = [vocab["<unk>"]]
    return torch.tensor(token_ids, dtype=torch.long)

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


In [6]:
# Dataset and batch collation.
# The Dataset turns each CSV row into token-id tensors.
# The collate function pads each batch to the longest sequence in the batch and
# returns the true lengths needed by the BiLSTM encoder.

class NLIDataset(Dataset):
    """PyTorch dataset for premise-hypothesis pairs."""
    def __init__(self, dataframe: pd.DataFrame, vocab: dict, max_len: int = 128, with_labels: bool = True):
        self.df = dataframe.reset_index(drop=True).copy()
        self.vocab = vocab
        self.max_len = max_len
        self.with_labels = with_labels

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        premise_ids = encode_text(row["premise"], self.vocab, self.max_len)
        hypothesis_ids = encode_text(row["hypothesis"], self.vocab, self.max_len)

        if self.with_labels:
            label = torch.tensor(float(row["label"]), dtype=torch.float32)
            return premise_ids, hypothesis_ids, label

        return premise_ids, hypothesis_ids

def nli_collate(batch):
    """Pad a batch of variable-length sequences and return true lengths."""
    if len(batch[0]) == 3:
        premises, hypotheses, labels = zip(*batch)
    else:
        premises, hypotheses = zip(*batch)
        labels = None

    premise_lengths = torch.tensor([len(x) for x in premises], dtype=torch.long)
    hypothesis_lengths = torch.tensor([len(x) for x in hypotheses], dtype=torch.long)

    premises = pad_sequence(premises, batch_first=True, padding_value=0)
    hypotheses = pad_sequence(hypotheses, batch_first=True, padding_value=0)

    if labels is None:
        return premises, premise_lengths, hypotheses, hypothesis_lengths

    labels = torch.stack(labels)
    return premises, premise_lengths, hypotheses, hypothesis_lengths, labels



In [7]:

def build_model(vocab_size, embedding_dim, hidden_size, padding_idx, embedding_matrix, dropout, train_embeddings):
    return ESIMPlus(
        vocab_size=vocab_size,
        embedding_dim=embedding_dim,
        hidden_size=hidden_size,
        padding_idx=padding_idx,
        embedding_matrix=embedding_matrix,
        dropout=dropout,
        train_embeddings=train_embeddings,
    ).to(device)


In [8]:

bundle = torch.load(MODEL_BUNDLE_PATH, map_location=device)
vocab = bundle["vocab"]
model_config = bundle["model_config"]
LOWERCASE = bool(model_config["lowercase"])

model = build_model(
    vocab_size=len(vocab),
    embedding_dim=model_config["embedding_dim"],
    hidden_size=model_config["hidden_size"],
    padding_idx=vocab["<pad>"],
    embedding_matrix=None,
    dropout=model_config["dropout"],
    train_embeddings=model_config["train_embeddings"],
)
model.load_state_dict(bundle["model_state_dict"])
model.eval()

best_threshold = float(bundle["best_threshold"])
print("Loaded model bundle from:", MODEL_BUNDLE_PATH)
print("Using prediction threshold:", best_threshold)

input_df = pd.read_csv(INPUT_PATH)
required_cols = {"premise", "hypothesis"}
if not required_cols.issubset(set(input_df.columns)):
    raise ValueError("Input CSV must contain 'premise' and 'hypothesis' columns.")

test_dataset = NLIDataset(input_df, vocab=vocab, max_len=model_config["max_len"], with_labels=False)
test_loader = DataLoader(test_dataset, batch_size=model_config["batch_size"], shuffle=False, collate_fn=nli_collate)

all_probs = []
with torch.no_grad():
    for premise_ids, premise_lengths, hypothesis_ids, hypothesis_lengths in test_loader:
        premise_ids = premise_ids.to(device)
        premise_lengths = premise_lengths.to(device)
        hypothesis_ids = hypothesis_ids.to(device)
        hypothesis_lengths = hypothesis_lengths.to(device)

        logits = model(premise_ids, premise_lengths, hypothesis_ids, hypothesis_lengths)
        probs = torch.sigmoid(logits)
        all_probs.extend(probs.cpu().numpy().tolist())

predictions = (np.asarray(all_probs) >= best_threshold).astype(int)
pd.DataFrame({"prediction": predictions}).to_csv(OUTPUT_PATH, index=False)
print(f"Saved predictions to {OUTPUT_PATH}")


Loaded model bundle from: nli_esim_plus_bundle.pt
Using prediction threshold: 0.54
Saved predictions to Group_n_B.csv
